# Train a diffusion model on your own images

In this notebook you train a small diffusion model **from scratch** on a set of images you choose, then generate new images in that style and download them.

You can use:
- **a dataset from Hugging Face** (just paste its name), or
- **your own images**, uploaded as a `.zip` file or read from a Google Drive folder.

**Before you start:** Runtime → Change runtime type → choose a **GPU** (A100 or G4 is fastest; T4 works but is slower).
Then run the cells **from top to bottom**.

### Tips for choosing images
- **Keep the subject consistent:** 300 photos of one kind of thing (flowers, album covers, your sketches) work much better than 300 random photos.
- **Centred and similar framing** helps a lot: the images are cropped to a square from the centre.
- **At least ~200 images.** Fewer works, but the model will start to copy them.
- The model works at **64×64 pixels**, so fine detail is lost. Bold shapes and colours survive best.

## 0. Setup

Colab sometimes ships a version of `torchao` that breaks `diffusers`. This cell checks for that problem and fixes it.

If it prints *“Fixing a package conflict…”*, the session restarts and Colab shows a “crashed” message. **That is expected.** Run this cell again, then continue.

In [ ]:
try:
    from diffusers import UNet2DModel
    print("diffusers is ready ✓")
except Exception as e:
    print("Fixing a package conflict, the session will restart…")
    import subprocess, os
    subprocess.run(["pip", "uninstall", "-y", "torchao"])
    os.kill(os.getpid(), 9)

## 1. Settings

**Choose where your images come from**, then adjust training if you want. Everything you might change is in this cell.

- `"huggingface"`: set `HF_DATASET` to a dataset name, e.g. `"huggan/smithsonian_butterflies_subset"`, `"huggan/flowers-102-categories"` or `"huggan/wikiart"`
- `"upload"`: you’ll be asked to upload a **.zip** of images (JPG or PNG)
- `"drive"`: set `DRIVE_FOLDER` to a folder of images in your Google Drive

**Training length is set in steps, not epochs**, so it works the same whether you have 200 or 20,000 images.

In [ ]:
import torch

# ── Where do the images come from? ──────────────────────────────
DATA_SOURCE  = "huggingface"            # "huggingface", "upload" or "drive"
HF_DATASET   = "huggan/smithsonian_butterflies_subset"
DRIVE_FOLDER = "/content/drive/MyDrive/my_images"
MAX_IMAGES   = 5000                     # use at most this many images

# ── Training ────────────────────────────────────────────────────
IMAGE_SIZE    = 64      # 64×64 pixels (32 trains faster, 128 is much slower)
BATCH_SIZE    = 32      # images per training step
TRAIN_STEPS   = 5000    # total training steps (more = better, slower)
LEARNING_RATE = 2e-4    # peak learning rate
WARMUP_STEPS  = 500     # steps to ramp the learning rate up from zero
EMA_DECAY     = 0.999   # how smooth the averaged (EMA) weights are
PREVIEW_EVERY = 1000    # show sample images every N steps
SEED          = 0

# ── Output ──────────────────────────────────────────────────────
NUM_TO_GENERATE = 16    # how many images to generate and download at the end
OUTPUT_NAME     = "my_diffusion"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
torch.manual_seed(SEED)

print("Device:", torch.cuda.get_device_name() if device.type == "cuda" else "CPU (training will be very slow!)")
print("bfloat16 speed-up:", "on" if USE_BF16 else "off (this GPU doesn’t support it)")

## 2. Load your images

This cell loads the images from the source you chose and finds the column that contains them.

In [ ]:
from datasets import load_dataset, Image as HFImage
import os, zipfile

if DATA_SOURCE == "huggingface":
    dataset = load_dataset(HF_DATASET, split="train")

elif DATA_SOURCE == "upload":
    from google.colab import files
    print("Choose a .zip file of images:")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content/my_images")
    dataset = load_dataset("imagefolder", data_dir="/content/my_images", split="train")

elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    dataset = load_dataset("imagefolder", data_dir=DRIVE_FOLDER, split="train")

else:
    raise ValueError('DATA_SOURCE must be "huggingface", "upload" or "drive"')

# Find the column that holds the images
image_col = next(name for name, feature in dataset.features.items() if isinstance(feature, HFImage))

dataset = dataset.shuffle(seed=SEED).select(range(min(MAX_IMAGES, len(dataset))))
print(f"Found {len(dataset)} images (column: '{image_col}')")
if len(dataset) < 100:
    print("⚠️ That’s a small dataset: expect the model to copy your images rather than invent new ones.")

## 3. Prepare the images

Each image is **cropped** to a square from the centre (not stretched), shrunk to 64×64 and flipped at random, which gives the model a little extra variety for free.

In [ ]:
from torchvision import transforms
from diffusers.utils import make_image_grid
from PIL import Image

def make_preprocess(flip=True):
    steps = [transforms.Resize(IMAGE_SIZE), transforms.CenterCrop(IMAGE_SIZE)]
    if flip:
        steps.append(transforms.RandomHorizontalFlip())
    steps += [transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]   # pixels → [-1, 1]
    return transforms.Compose(steps)

preprocess = make_preprocess(flip=True)

def transform(batch):
    return {"pixel_values": [preprocess(img.convert("RGB")) for img in batch[image_col]]}

train_dataset = dataset.with_transform(transform)

batch_size = min(BATCH_SIZE, len(train_dataset))       # small datasets: never ask for more than we have
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    persistent_workers=True,
    collate_fn=lambda b: torch.stack([x["pixel_values"] for x in b]),
)

def to_pil(batch, size=128):
    """Tensors in [-1, 1] → list of (enlarged) images for display."""
    batch = ((batch.clamp(-1, 1) + 1) / 2 * 255).byte().cpu()
    return [Image.fromarray(img.permute(1, 2, 0).numpy()).resize((size, size), Image.NEAREST)
            for img in batch]

epochs = -(-TRAIN_STEPS // len(train_dataloader))    # round up
print(f"{TRAIN_STEPS} steps = about {epochs} passes through your {len(train_dataset)} images")

# This is what the model will learn from
preview = next(iter(train_dataloader))[:8]
make_image_grid(to_pil(preview), rows=1, cols=len(preview))

## 4. Model, noise schedule, optimizer

- **UNet**: the network that looks at a noisy image and predicts the noise in it.
- **Noise scheduler**: how much noise is added at each of the 1,000 steps. The cosine schedule (`squaredcos_cap_v2`) suits small images better than the default.
- **EMA**: a slowly updated average of the model’s weights. Smoother than the raw weights, and gives cleaner images.
- **Learning-rate schedule**: warm up for the first steps, then slowly decrease.

In [ ]:
from diffusers import UNet2DModel, DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_cosine_schedule_with_warmup

def make_model():
    return UNet2DModel(
        sample_size=IMAGE_SIZE,
        in_channels=3,
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(64, 128, 256, 512),
        down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    ).to(device)

model = make_model()
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} million parameters")

noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")

ema = EMAModel(model.parameters(), decay=EMA_DECAY)
ema.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
lr_scheduler = get_cosine_schedule_with_warmup(optimizer, min(WARMUP_STEPS, TRAIN_STEPS // 10), TRAIN_STEPS)

## 5. A function to generate images

We use **DDIM** to generate: it reaches similar quality in 50–100 steps instead of 1,000.
The model is the same; only the way we walk back from noise to image changes.

In [ ]:
from diffusers import DDIMScheduler

@torch.no_grad()
def generate(unet, n=8, steps=100, seed=0):
    """Generate n images. Returns tensors in [-1, 1] (use to_pil to view them)."""
    sampler = DDIMScheduler.from_config(noise_scheduler.config)   # same noise schedule as training
    sampler.set_timesteps(steps)
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, generator=generator, device=device)  # start: pure noise
    was_training = unet.training
    unet.eval()
    for t in sampler.timesteps:
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_BF16):
            noise_pred = unet(x, t).sample                         # predict the noise
        x = sampler.step(noise_pred.float(), t, x).prev_sample     # remove a little of it
    unet.train(was_training)
    return x.clamp(-1, 1)

def preview_with_ema(n=8, steps=50):
    """Generate with the smoothed (EMA) weights, then switch back to the training weights."""
    ema.store(model.parameters())
    ema.copy_to(model.parameters())
    images = to_pil(generate(model, n=n, steps=steps))
    ema.restore(model.parameters())
    return make_image_grid(images, rows=1, cols=n)

## 6. Train

Each step: take real images, add a random amount of noise, ask the model to predict that noise, and correct it based on how wrong it was.

Every `PREVIEW_EVERY` steps you’ll see samples generated with the EMA weights. Watch them go from blobs to your subject.

**Timing (roughly):** 5,000 steps take about 5 minutes on an A100 or G4, and 20+ minutes on a T4.

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display

losses = []
step = 0
model.train()
progress = tqdm(total=TRAIN_STEPS, desc="Training")

while step < TRAIN_STEPS:
    for clean_images in train_dataloader:
        clean_images = clean_images.to(device, non_blocking=True)

        # 1. Add a random amount of noise to each image
        noise = torch.randn_like(clean_images)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (clean_images.shape[0],), device=device).long()
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        # 2. Ask the model to predict that noise
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_BF16):
            noise_pred = model(noisy_images, timesteps).sample
        loss = F.mse_loss(noise_pred.float(), noise)

        # 3. Learn from the mistake
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # avoid sudden huge updates
        optimizer.step()
        lr_scheduler.step()
        ema.step(model.parameters())                              # update the smoothed copy

        losses.append(loss.item())
        step += 1
        progress.update(1)
        progress.set_postfix(loss=f"{loss.item():.4f}")

        if step % PREVIEW_EVERY == 0 or step == TRAIN_STEPS:
            progress.write(f"Step {step}: average loss {sum(losses[-100:]) / len(losses[-100:]):.4f}")
            display(preview_with_ema())
        if step >= TRAIN_STEPS:
            break

progress.close()

## 7. Training loss

The loss drops fast, then flattens. **Flat doesn’t mean finished:** image quality usually keeps improving after the curve looks flat.

In [ ]:
from matplotlib import pyplot as plt

def smooth(values, k=50):
    return [sum(values[max(0, i - k):i + 1]) / len(values[max(0, i - k):i + 1]) for i in range(len(values))]

plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, label="each step")
plt.plot(smooth(losses), label="smoothed")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Training loss")
plt.legend()
plt.show()

## 8. Save the model

We save the **EMA weights**, the smoothed version that generates cleaner images.

Files in Colab disappear when the session ends. To keep the model, uncomment the Google Drive lines.

In [ ]:
ema.copy_to(model.parameters())                   # replace training weights with the EMA weights
model_file = f"{OUTPUT_NAME}_ema.pth"
torch.save(model.state_dict(), model_file)
print("Saved", model_file)

# from google.colab import drive
# drive.mount("/content/drive")
# torch.save(model.state_dict(), f"/content/drive/MyDrive/{model_file}")

## 9. Generate, save and download

This cell loads the saved model, generates `NUM_TO_GENERATE` images, shows them, and downloads them as a `.zip`: each image as its own PNG, plus a grid of all of them.

Change `seed` for a different set, or `steps` to see how quality changes with fewer steps. `size` sets how large the saved images are; they are enlarged from 64×64, so they will look soft or pixelated.

In [ ]:
import shutil

trained = make_model()
trained.load_state_dict(torch.load(model_file, map_location=device, weights_only=True))

generated = generate(trained, n=NUM_TO_GENERATE, steps=100, seed=42)
images = to_pil(generated, size=256)

# Save each image, plus a grid
out_dir = f"{OUTPUT_NAME}_images"
os.makedirs(out_dir, exist_ok=True)
for i, img in enumerate(images):
    img.save(f"{out_dir}/{OUTPUT_NAME}_{i:02d}.png")
cols = min(8, len(images))
rows = -(-len(images) // cols)                                    # round up
blanks = [Image.new("RGB", images[0].size, "white")] * (rows * cols - len(images))   # fill the last row
grid = make_image_grid(images + blanks, rows=rows, cols=cols)
grid.save(f"{out_dir}/{OUTPUT_NAME}_grid.png")

# Zip and download
zip_path = shutil.make_archive(out_dir, "zip", out_dir)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Saved", zip_path)

grid

## 10. Is the model inventing or copying?

A model trained on few images can **memorise** them and hand back near-copies. This cell puts each generated image next to the **most similar image in your training set** (by pixel difference).

- If the pairs look almost identical, the model is copying: use more images or fewer training steps.
- If they only share a style, the model is generating something new.

This matters beyond the classroom: memorisation is one of the central questions in debates about AI and copyright.

In [ ]:
check = make_preprocess(flip=False)
n_check = min(len(dataset), 2000)
train_tensors = torch.stack([check(dataset[i][image_col].convert("RGB")) for i in range(n_check)]).to(device)

n_show = min(8, len(generated))
distances = torch.cdist(generated[:n_show].flatten(1), train_tensors.flatten(1))   # pixel distance to every training image
nearest = distances.argmin(dim=1)

pairs = []
for g_img, idx in zip(to_pil(generated[:n_show]), nearest.tolist()):
    pairs += [g_img, to_pil(train_tensors[idx:idx + 1])[0]]

print("Top row: generated · Bottom row: closest training image")
make_image_grid(pairs[0::2] + pairs[1::2], rows=2, cols=n_show)

## 11. Experiments to try

Change one setting at a time in section 1, retrain, and compare:

- **Fewer images:** `MAX_IMAGES = 100`. Run section 10: does the model start copying?
- **Shorter training:** `TRAIN_STEPS = 1000`. Compare the previews with 5,000 steps.
- **Smaller images:** `IMAGE_SIZE = 32`. Much faster. What is still recognisable?
- **Fewer generation steps:** in section 9, try `steps=10` and `steps=200`.
- **Mixed datasets:** combine two very different sets of images in one folder. Does the model blend them, or switch between them?